# KVQuant Implementation -- full-precision baseline only

One of four split-out notebooks (2-bit / 3-bit / 4-bit / full-precision
baseline), each running independently so they can execute in parallel
Colab sessions instead of one long combined run. This notebook loads and
evaluates ONLY the untouched, unquantized full-precision model -- no
KVCacheCompression repo clone, no patches, no Fisher calibration, no
Quantize step, since none of that machinery applies to a model that never
gets quantized. The three quantized bit-widths each live in their own
dedicated notebook (`KVQuant_2bit_Implementation.ipynb`,
`_3bit_`, `_4bit_`), so this baseline only needs to run once rather than
being redundantly repeated in all three.

Every fix already built into v3 is preserved as-is: teacher-forced
step-by-step TTFT/TBT timing for PTB/WikiText-103, real
`generate()`-based GSM8K timing with TBT excluding TTFT, H2O-matching
sampling (full-text-then-chunk for PTB/WikiText-103, sequential first
`N_EVAL_SAMPLES` questions for GSM8K, no shuffling), and
download-retry-with-cache-clear robustness for every network call. Memory
is measured directly from the real KV cache tensors this model actually
produces (no outlier-hook simulation needed, since nothing here is
quantized).

Run cells top to bottom. Needs a GPU runtime.

In [ ]:
# Block 1 - Environment setup
# Run once per fresh runtime. Package versions are pinned so environment
# differences are never a confound between compression methods (kept
# identical to the 2-bit/3-bit/4-bit notebooks even though this one never
# clones/patches the KVCacheCompression repo, so there's no risk of a
# transformers/tokenizers/etc. version mismatch skewing the comparison).

import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

from google.colab import drive
drive.mount("/content/drive")

!python -m pip install -q --no-deps \
  "transformers==4.43.4" \
  "accelerate==0.33.0" \
  "tokenizers==0.19.1" \
  "huggingface_hub==0.36.2" \
  sentencepiece \
  einops

!python -m pip install -q \
  "datasets==2.14.5" \
  tqdm \
  matplotlib

!python -m pip install -q --no-deps --force-reinstall "huggingface_hub==0.36.2"

import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

try:
    from google.colab import userdata
    _hf_token = userdata.get("HF_TOKEN")
except Exception:
    _hf_token = os.environ.get("HF_TOKEN")

if _hf_token:
    from huggingface_hub import login
    login(token=_hf_token)
    print("Logged in to HuggingFace")
else:
    print("No HF_TOKEN found -- skipping login (fine for public models like TinyLlama)")

print("Block 1 finished. Now run Block 2.")

In [ ]:
# Block 2 - Imports, GPU check

import gc
import math
import os
import re
import shutil
import time
import random
import pickle
import sys

import numpy as np
import torch
import torch.nn as nn
import pandas as pd

import datasets
import transformers
import huggingface_hub
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA")

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
MODEL_DTYPE = torch.float16 if HAS_CUDA else torch.float32


def sync_if_cuda():
    if HAS_CUDA:
        torch.cuda.synchronize()


def clear_memory():
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()


def clear_hf_dataset_cache(*dataset_names):
    """Removes cached files for the given HF dataset repo name(s) (e.g.
    "wikitext", "gsm8k") from both the datasets cache and the hub cache.
    Used as an on_retry hook: a download that breaks partway through can
    leave a corrupted partial file that every subsequent retry just
    resumes (and re-breaks at the same point) instead of truly restarting
    -- clearing the cache forces a genuinely fresh download."""
    home = os.path.expanduser("~")
    for name in dataset_names:
        for base in [
            os.path.join(home, ".cache", "huggingface", "datasets", name),
            os.path.join(home, ".cache", "huggingface", "hub", f"datasets--{name}"),
        ]:
            shutil.rmtree(base, ignore_errors=True)


def robust_call(fn, *args, max_retries=5, backoff_sec=5, desc="operation", on_retry=None, **kwargs):
    """Retries fn(*args, **kwargs) on any exception, up to max_retries times,
    waiting backoff_sec between attempts -- guards dataset downloads against
    transient network failures (e.g. IncompleteRead/ChunkedEncodingError)
    rather than letting one flaky connection kill the whole notebook run.
    If on_retry is given, it's called (no args) after each failure, before
    the next attempt -- e.g. clear_hf_dataset_cache, so a retry that hit a
    stuck/corrupted partial download actually starts fresh instead of
    resuming (and re-breaking at) the same point every time."""
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            _msg = f"  {desc}: attempt {attempt}/{max_retries} failed ({e!r})"
            if attempt < max_retries:
                _msg += f", retrying in {backoff_sec}s..."
            print(_msg)
            if attempt < max_retries:
                if on_retry is not None:
                    on_retry()
                time.sleep(backoff_sec)
    raise last_err


if not HAS_CUDA:
    print("WARNING: No GPU detected. This will be very slow.")

In [ ]:
# Block 3 - Experiment settings.
# Kept identical in spirit to H2O_Implementation_v3.ipynb wherever the same
# quantity applies (C, SHARED_SEED, GSM8K prompt, N_EVAL_SAMPLES,
# GSM8K_MAX_NEW_TOKENS) so the two methods' evaluations are only different
# where the compression method itself makes them different. No
# ABITS/SPARSITY_THRESHOLD/quantizer settings here -- this notebook never
# quantizes anything.

LOCAL_MODEL_PATH = "/content/tinyllama-1.1b"
HF_MODEL_ID = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
MODEL_ID = LOCAL_MODEL_PATH if os.path.exists(LOCAL_MODEL_PATH) else HF_MODEL_ID

SHARED_SEED = 0
C = 2048                    # chunk length for PTB/WikiText-103 = model's max context
N_EVAL_SAMPLES = 64          # GSM8K question count; also caps PTB/WikiText-103 source prompts
GSM8K_MAX_NEW_TOKENS = 256
METHOD_NAME = "kvquant_baseline_full_precision"

random.seed(SHARED_SEED)
torch.manual_seed(SHARED_SEED)

GSM8K_FEWSHOT_PREFIX = (
    "You are solving grade-school math word problems.\n"
    "Show the calculation briefly, then end with exactly this format:\n"
    "#### <final number>\n\n"
    "Question: Mia has 3 marbles and buys 4 more. How many marbles does she have?\n"
    "Answer: Mia has 3 + 4 = 7 marbles.\n"
    "#### 7\n\n"
    "Question: A box has 6 pencils. Sam buys 5 boxes. How many pencils does Sam buy?\n"
    "Answer: Sam buys 6 * 5 = 30 pencils.\n"
    "#### 30\n"
)

print("Model:", MODEL_ID)
print("Method:", METHOD_NAME)
print("Chunk length C:", C)
print("N_EVAL_SAMPLES (GSM8K):", N_EVAL_SAMPLES)
print("GSM8K max new tokens:", GSM8K_MAX_NEW_TOKENS)

In [ ]:
# Block - Load tokenizer + the untouched full-precision model. No repo
# clone, no patches, no Fisher calibration, no Quantize step -- none of
# that machinery is needed for a model that's never quantized. This is the
# exact same loading code the combined v3 notebook used for model_fp,
# unchanged, so the baseline is byte-for-byte comparable across notebooks.

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

_model_kwargs = {
    "torch_dtype": MODEL_DTYPE,
    "low_cpu_mem_usage": True,
    "attn_implementation": "sdpa",
    "trust_remote_code": True,
}
if HAS_CUDA:
    _model_kwargs["device_map"] = {"": 0}

print("Loading full-precision baseline model...")
model_fp = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_model_kwargs)
if not HAS_CUDA:
    model_fp = model_fp.to(DEVICE)
model_fp.eval()
model_fp.config.use_cache = True

device = next(model_fp.parameters()).device
print("model_fp: full-precision baseline, loaded and ready.")

In [ ]:
# Block - PTB / WikiText-103: load the FULL test text, concatenate ALL of
# it into one long sequence, tokenize, split into consecutive chunks of
# length C, then cap to the FIRST N_EVAL_SAMPLES chunks -- a deterministic
# prefix of the real test set, not a random sample. Matches H2O's
# PTB/WikiText-103 loading exactly, so the two methods are evaluated on
# identical data. Both downloads go through robust_call so a transient
# network blip gets retried instead of killing the whole run; WikiText-103's
# load also clears its HF cache between retries (on_retry), since a broken
# 157MB download can leave a corrupted partial file that plain retries
# just resume (and re-break at the same point) forever.


def chunk_sequence(token_ids_1d, chunk_len):
    chunks = []
    n = token_ids_1d.shape[0]
    for start in range(0, n, chunk_len):
        chunks.append(token_ids_1d[start:start + chunk_len])
    return chunks


def load_ptb_chunks():
    import urllib.request
    test_url = "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb.test.txt"

    def _fetch():
        return urllib.request.urlopen(test_url).read().decode("utf-8")

    test_text_raw = robust_call(_fetch, desc="PTB test.txt download")
    sentences = [s.strip() for s in test_text_raw.splitlines() if s.strip()]
    full_text = "\n\n".join(sentences)
    ids = tokenizer(full_text, return_tensors="pt")["input_ids"][0]
    print(f"PTB test set: {len(sentences)} lines, {ids.shape[0]} tokens total")
    return chunk_sequence(ids, C)[:N_EVAL_SAMPLES]


def load_wikitext103_chunks():
    testdata = robust_call(
        load_dataset, "wikitext", "wikitext-103-raw-v1", split="test",
        desc="WikiText-103 test load", on_retry=lambda: clear_hf_dataset_cache("wikitext"),
    )
    texts = [t for t in testdata["text"] if len(t.strip()) > 0]
    full_text = "\n\n".join(texts)
    ids = tokenizer(full_text, return_tensors="pt")["input_ids"][0]
    print(f"WikiText-103 test set: {len(texts)} non-empty lines, {ids.shape[0]} tokens total")
    return chunk_sequence(ids, C)[:N_EVAL_SAMPLES]


ptb_chunks = load_ptb_chunks()
wikitext103_chunks = load_wikitext103_chunks()
print(f"PTB: {len(ptb_chunks)} chunks of up to {C} tokens (first N_EVAL_SAMPLES={N_EVAL_SAMPLES}, not randomly selected)")
print(f"WikiText-103: {len(wikitext103_chunks)} chunks of up to {C} tokens (first N_EVAL_SAMPLES={N_EVAL_SAMPLES}, not randomly selected)")

In [ ]:
# Block - KV-cache memory (measured from REAL cache tensors) + step-by-
# step-per-chunk eval function.
#
# TTFT/TBT/perplexity/accuracy come ONLY from the timed step-by-step loop
# below -- the model is stepped one token at a time using its own KV cache,
# teacher-forced (real corpus tokens fed in, never the model's own
# prediction). Memory is measured in a SEPARATE, UNTIMED pass
# (measure_chunk_kv_memory) run AFTER the timed loop finishes, so it cannot
# affect any timing number.
#
# No outlier-hook tracker exists here (nothing is quantized), so memory is
# measured directly: an untimed forward pass with use_cache=True, then
# summing the real byte size of the real past_key_values tensors it
# produced.


@torch.no_grad()
def measure_chunk_kv_memory(model_obj, chunk_ids_1d, bits_per_element, tracker=None):
    chunk_ids = chunk_ids_1d.unsqueeze(0).to(device)

    if tracker is not None:
        reset_outlier_tracker(tracker)
        model_obj(input_ids=chunk_ids, use_cache=False, return_dict=True)
        return measure_bytes_from_tracker(tracker, bits_per_element)

    outputs = model_obj(input_ids=chunk_ids, use_cache=True, return_dict=True)
    pkv = outputs.past_key_values
    legacy = pkv.to_legacy_cache() if hasattr(pkv, "to_legacy_cache") else pkv
    total_bytes = 0
    for layer_kv in legacy:
        for t in layer_kv:
            total_bytes += t.numel() * t.element_size()
    return total_bytes


@torch.no_grad()
def score_chunk_kvquant(model_obj, chunk_ids_1d, bits_per_element, tracker=None):
    L = chunk_ids_1d.shape[0]
    if L < 2:
        return None
    chunk_ids = chunk_ids_1d.unsqueeze(0).to(device)
    input_ids = chunk_ids[:, :-1]
    target_ids = chunk_ids[:, 1:]

    loss_fct = nn.CrossEntropyLoss()
    nll_sum = 0.0
    correct = 0
    scored = 0
    step_times = []
    past_key_values = None

    for pos in range(input_ids.shape[1]):
        step_input = input_ids[:, pos:pos + 1]

        sync_if_cuda()
        t0 = time.perf_counter()
        outputs = model_obj(
            input_ids=step_input, past_key_values=past_key_values,
            use_cache=True, return_dict=True,
        )
        sync_if_cuda()
        step_times.append(time.perf_counter() - t0)

        past_key_values = outputs.past_key_values
        step_logits = outputs.logits[:, -1, :]
        step_target = target_ids[:, pos]

        loss = loss_fct(step_logits, step_target)
        nll_sum += loss.float().item()

        pred = step_logits.argmax(dim=-1)
        correct += int((pred == step_target).sum().item())
        scored += 1

    ttft_sec = step_times[0]
    tbt_sec = sum(step_times[1:]) / len(step_times[1:]) if len(step_times) > 1 else 0.0
    total_latency_sec = sum(step_times)

    peak_bytes = measure_chunk_kv_memory(model_obj, chunk_ids_1d, bits_per_element, tracker)

    return {
        "nll_sum": nll_sum, "scored": scored, "correct": correct,
        "ttft_sec": ttft_sec, "tbt_sec": tbt_sec,
        "total_latency_sec": total_latency_sec, "peak_memory_bytes": peak_bytes,
    }


@torch.no_grad()
def preview_chunk_prediction(model_obj, chunk_ids_1d, n_preview_tokens=30):
    """Untimed, separate forward pass over just the first n_preview_tokens of
    a chunk -- for printing what the model actually predicted vs. the real
    next tokens. Does not affect any reported metric."""
    n_preview_tokens = min(n_preview_tokens, chunk_ids_1d.shape[0] - 1)
    if n_preview_tokens < 1:
        return None
    preview_ids = chunk_ids_1d[:n_preview_tokens + 1].unsqueeze(0).to(device)
    input_ids = preview_ids[:, :-1]
    target_ids = preview_ids[:, 1:]

    outputs = model_obj(input_ids=input_ids, use_cache=False, return_dict=True)
    preds = outputs.logits.argmax(dim=-1)

    return {
        "prompt_text": tokenizer.decode(input_ids[0], skip_special_tokens=True),
        "actual_next_text": tokenizer.decode(target_ids[0], skip_special_tokens=True),
        "predicted_next_text": tokenizer.decode(preds[0], skip_special_tokens=True),
    }

In [ ]:
# Block - PTB / WikiText-103 driver: run every chunk through
# score_chunk_kvquant against the untouched full-precision model.


def evaluate_lm_dataset_kvquant(dataset_name, chunks, model_obj, bits_per_element, method_label, tracker=None):
    clear_memory()
    total_nll = 0.0
    total_scored = 0
    total_correct = 0
    ttft_values, tbt_values, latency_values, peak_mem_values = [], [], [], []

    N_PREVIEW_CHUNKS = 5
    for chunk_idx, chunk_ids in enumerate(tqdm(chunks, desc=f"{dataset_name} | {method_label}")):
        if chunk_idx < N_PREVIEW_CHUNKS:
            preview = preview_chunk_prediction(model_obj, chunk_ids)
            if preview is not None:
                print(f"\n--- {dataset_name} | {method_label} | chunk {chunk_idx} preview ---")
                print(f"Prompt:              {preview['prompt_text']!r}")
                print(f"Actual next text:    {preview['actual_next_text']!r}")
                print(f"Predicted next text: {preview['predicted_next_text']!r}")

        result = score_chunk_kvquant(model_obj, chunk_ids, bits_per_element, tracker)
        if result is None:
            continue
        total_nll += result["nll_sum"]
        total_scored += result["scored"]
        total_correct += result["correct"]
        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])

    avg_nll = total_nll / max(total_scored, 1)
    perplexity = math.exp(min(avg_nll, 50.0))
    accuracy = total_correct / max(total_scored, 1)

    return {
        "dataset": dataset_name,
        "method": method_label,
        "perplexity": perplexity,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


lm_results = []
for _name, _chunks in [("PTB", ptb_chunks), ("WikiText-103", wikitext103_chunks)]:
    lm_results.append(evaluate_lm_dataset_kvquant(_name, _chunks, model_fp, 16, METHOD_NAME))

lm_results_df = pd.DataFrame(lm_results)
display(lm_results_df)

In [ ]:
# Block - GSM8K loading: canonical source, sequential first N_EVAL_SAMPLES
# questions (no shuffling), few-shot prompt -- identical format to
# H2O_Implementation_v3.ipynb so the two methods' GSM8K prompts and
# question sets can never diverge. Download goes through robust_call,
# clearing its HF cache between retries, so a transient network blip or
# stuck partial download gets retried cleanly instead of killing the whole
# run.


def extract_gsm8k_gold_answer(answer_text):
    m = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", answer_text)
    if not m:
        return None
    try:
        return float(m.group(1).replace(",", ""))
    except ValueError:
        return None


gsm8k_test = robust_call(
    load_dataset, "gsm8k", "main", split="test",
    desc="GSM8K test load", on_retry=lambda: clear_hf_dataset_cache("gsm8k"),
)

gsm8k_qa_pairs = []
for i in range(min(N_EVAL_SAMPLES, len(gsm8k_test))):
    item = gsm8k_test[i]
    gold = extract_gsm8k_gold_answer(item["answer"])
    if gold is not None:
        gsm8k_qa_pairs.append({"question": item["question"], "gold": gold, "gold_text": item["answer"]})

print(f"GSM8K: {len(gsm8k_qa_pairs)} questions loaded (of {len(gsm8k_test)} total in test set)")

In [ ]:
# Block - GSM8K generation with the full-precision baseline. Uses the real
# model.generate() call -- prefill processes the whole prompt in one shot,
# then decode continues one token at a time exactly like a normal
# generate() loop. A StoppingCriteria timestamps every generated token so
# TTFT/TBT come from this SAME call that produces the graded answer -- not
# a separate probe. Memory is measured in a SEPARATE, untimed pass after
# generate() returns, so it cannot affect any timing number.


def _extract_final_number(text):
    m = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", text)
    if m:
        num_str = m.group(1)
    else:
        nums = re.findall(r"-?[0-9][0-9,]*\.?[0-9]*", text)
        if not nums:
            return None
        num_str = nums[-1]
    num_str = num_str.replace(",", "").rstrip(".")
    try:
        return float(num_str)
    except ValueError:
        return None


class _TimingCriteria(StoppingCriteria):
    def __init__(self):
        self.token_times = []

    def __call__(self, input_ids, scores, **kwargs):
        self.token_times.append(time.perf_counter())
        return False


@torch.no_grad()
def generate_gsm8k_kvquant(model_obj, question, bits_per_element, tracker=None):
    prompt = GSM8K_FEWSHOT_PREFIX + f"\nQuestion: {question.strip()}\nAnswer:"
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = enc["input_ids"].shape[1]

    timing = _TimingCriteria()
    sync_if_cuda()
    gen_start = time.perf_counter()
    gen_ids = model_obj.generate(
        **enc, max_new_tokens=GSM8K_MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        stopping_criteria=StoppingCriteriaList([timing]),
    )
    sync_if_cuda()
    gen_end = time.perf_counter()
    total_latency_sec = gen_end - gen_start

    gen_text = tokenizer.decode(gen_ids[0][prompt_len:], skip_special_tokens=True)
    gen_text = gen_text.split("Question:")[0]

    n_generated = len(timing.token_times)
    if n_generated > 0:
        ttft_sec = timing.token_times[0] - gen_start
    else:
        ttft_sec = total_latency_sec
    tbt_sec = (total_latency_sec - ttft_sec) / max(n_generated - 1, 1)

    total_tokens = prompt_len + n_generated

    # Untimed pass over the full generated sequence, purely to measure
    # memory -- real cache tensor bytes for the full-precision baseline
    # (no outlier-hook tracker needed, nothing here is quantized). Does
    # not affect the timed generate() call above.
    if tracker is not None:
        reset_outlier_tracker(tracker)
        model_obj(input_ids=gen_ids, use_cache=False, return_dict=True)
        peak_bytes = measure_bytes_from_tracker(tracker, bits_per_element)
    else:
        cache_outputs = model_obj(input_ids=gen_ids, use_cache=True, return_dict=True)
        pkv = cache_outputs.past_key_values
        legacy = pkv.to_legacy_cache() if hasattr(pkv, "to_legacy_cache") else pkv
        peak_bytes = sum(t.numel() * t.element_size() for layer_kv in legacy for t in layer_kv)

    return {
        "gen_text": gen_text, "ttft_sec": ttft_sec, "tbt_sec": tbt_sec,
        "total_latency_sec": total_latency_sec, "total_tokens": total_tokens,
        "peak_memory_bytes": peak_bytes,
    }


@torch.no_grad()
def score_gsm8k_perplexity_kvquant(model_obj, question, gold_text):
    """Secondary, untimed teacher-forced pass on the gold answer -- per the
    metrics section: perplexity for GSM8K is usually calculated by teacher-
    forcing the correct answer, not by scoring the generated answer."""
    gold_number = extract_gsm8k_gold_answer(gold_text)
    if gold_number is None:
        return None
    gold_str = str(int(gold_number)) if gold_number == int(gold_number) else str(gold_number)
    prompt = GSM8K_FEWSHOT_PREFIX + f"\nQuestion: {question.strip()}\nAnswer:"
    prompt_ids = tokenizer(prompt, add_special_tokens=True)["input_ids"]
    target_ids = tokenizer(" " + gold_str, add_special_tokens=False)["input_ids"]
    full_ids = torch.tensor([prompt_ids + target_ids], device=device)

    outputs = model_obj(input_ids=full_ids, use_cache=True, return_dict=True)
    logits = outputs.logits[0]
    n_prompt = len(prompt_ids)
    nll_sum = 0.0
    scored = 0
    for ti, target_id in enumerate(target_ids):
        pos = n_prompt - 1 + ti
        if pos < 0 or pos >= logits.shape[0]:
            continue
        log_probs = torch.log_softmax(logits[pos].float(), dim=-1)
        nll_sum += -log_probs[target_id].item()
        scored += 1
    if scored == 0:
        return None
    return math.exp(min(nll_sum / scored, 50.0))

In [ ]:
# Block - GSM8K driver: run every question through generate_gsm8k_kvquant
# (accuracy + TTFT/TBT/latency + memory, all from the SAME real generation
# call plus its untimed memory-measurement pass), plus the secondary
# teacher-forced perplexity pass, against the full-precision baseline.


def evaluate_gsm8k_kvquant(qa_pairs, model_obj, bits_per_element, method_label, tracker=None):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []

    N_PREVIEW_QUESTIONS = 5
    for q_idx, qa in enumerate(tqdm(qa_pairs, desc=f"GSM8K | {method_label}")):
        result = generate_gsm8k_kvquant(model_obj, qa["question"], bits_per_element, tracker)
        pred = _extract_final_number(result["gen_text"])
        is_correct = pred is not None and abs(pred - qa["gold"]) < 1e-4
        correct += int(is_correct)
        total += 1

        if q_idx < N_PREVIEW_QUESTIONS:
            print(f"\n--- GSM8K | {method_label} | question {q_idx} preview ---")
            print(f"Question:    {qa['question']}")
            print(f"Generated:   {result['gen_text'].strip()}")
            print(f"Gold answer: {qa['gold']} | Predicted: {pred} | Correct: {is_correct}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])

        ppl = score_gsm8k_perplexity_kvquant(model_obj, qa["question"], qa["gold_text"])
        if ppl is not None:
            ppl_values.append(ppl)

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    return {
        "dataset": "GSM8K",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


gsm8k_results = [
    evaluate_gsm8k_kvquant(gsm8k_qa_pairs, model_fp, 16, METHOD_NAME),
]
gsm8k_results_df = pd.DataFrame(gsm8k_results)
display(gsm8k_results_df)

In [ ]:
# Block - Combine PTB / WikiText-103 / GSM8K results into one table with
# ONLY the specified metrics, then save to CSV. This notebook only ever
# produces "kvquant_baseline_full_precision" rows -- 2-bit, 3-bit, and
# 4-bit are saved by their own separate notebooks, so there's no
# cross-notebook filename collision risk.

results_df = pd.concat([lm_results_df, gsm8k_results_df], ignore_index=True)
results_df = results_df[[
    "dataset", "method", "perplexity", "accuracy",
    "ttft_sec", "tbt_sec", "avg_total_latency_sec",
    "peak_memory_gb", "average_memory_gb",
]]
display(results_df)

os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)

_path = "/content/drive/MyDrive/KVQuant_v3_Results/kvquant_baseline_full_precision_results.csv"
results_df.to_csv(_path, index=False)
print(f"Saved to {_path}")